In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from category_encoders import TargetEncoder
from IPython.display import display

dane_raw = pd.read_csv("dane/otomoto_offers_eng_23-04-2023.csv", sep=";", encoding="utf-8")
df_almost_clean = dane_raw.copy()

print(f"Wymiary danych: {df_almost_clean.shape}")

In [ ]:
# zamiana pustych miejsc na NaN 
str_cols = df_almost_clean.select_dtypes(include=['object']).columns

for col in str_cols:
    df_almost_clean[col] = df_almost_clean[col].astype(str).str.strip()


df_almost_clean[str_cols] = df_almost_clean[str_cols].replace(["", "nan", "NaN"], np.nan)

print(f"Liczba wszystkich braków danych (NaN) przed czyszczeniem: {df_almost_clean.isna().sum().sum()}")
display(df_almost_clean.head(2))

In [ ]:
if 'fuel_type' in df_almost_clean.columns and 'engine_displacement' in df_almost_clean.columns:
    is_electric = df_almost_clean['fuel_type'] == 'Electric'
    # Wypełniamy tylko te komórki, które u elektryków były brakami danych 
    df_almost_clean.loc[is_electric, 'engine_displacement'] = df_almost_clean.loc[is_electric, 'engine_displacement'].fillna(0)

print(f"Liczba wszystkich braków danych (NaN) w całej bazie: {df_almost_clean.isna().sum().sum()}")

if 'fuel_type' in df_almost_clean.columns and 'engine_displacement' in df_almost_clean.columns:
    braki_elektryki = df_almost_clean[df_almost_clean['fuel_type'] == 'Electric']['engine_displacement'].isna().sum()
    print(f"Liczba braków w 'engine_displacement' dla aut elektrycznych: {braki_elektryki} (Powinno być 0)")

display(df_almost_clean.head(2))

In [ ]:

# Raport o zmiennych do excela
def generate_metadata_table(data, target_col="price"):
    report_rows = []
    total_rows = len(data)
    
    for col in data.columns:
        col_data = data[col]
        
        # 1. Podstawowe informacje
        v_type = str(col_data.dtype)
        
        # Liczenie braków
        na_count = col_data.isna().sum()
        empty_str_count = (col_data == "").sum() if col_data.dtype == 'object' else 0
        total_missings = na_count + empty_str_count
        missing_pct = (total_missings / total_rows) * 100
        
        # 2. Wykluczenie braków danych i pustych ciągów do statystyk
        if col_data.dtype == 'object':
            valid_data = col_data.dropna()
            valid_data = valid_data[valid_data != ""]
        else:
            valid_data = col_data.dropna()
            
        unique_count = valid_data.nunique()
        
        # Wyznaczanie mody  i jej udziału
        if len(valid_data) > 0:
            val_counts = valid_data.value_counts()
            top_value = val_counts.index[0]
            top_value_pct = (val_counts.iloc[0] / len(valid_data)) * 100
            
            # max 3 unikalne, niepuste wartości do podglądu
            sample_vals = "; ".join(map(str, valid_data.unique()[:3]))
        else:
            top_value = np.nan
            top_value_pct = 0
            sample_vals = "Same braki"
            
        # 3. Statystyki numeryczne i korelacja z ceną
        v_min = np.nan
        v_max = np.nan
        v_median = np.nan
        v_cor_with_target = np.nan
        
        # Sprawdzenie czy kolumna jest numeryczna
        if pd.api.types.is_numeric_dtype(col_data):
            if len(valid_data) > 0:
                v_min = valid_data.min()
                v_max = valid_data.max()
                v_median = valid_data.median()
                
                # Zabezpieczone wyliczanie korelacji 
                if col != target_col and target_col in data.columns:
                    try:
                        v_cor_with_target = col_data.corr(data[target_col], method='spearman')
                    except Exception:
                        v_cor_with_target = np.nan
                        
        # 4. Konstruowanie pojedynczego wiersza raportu
        report_rows.append({
            "Zmienna": col,
            "Typ_Danych": v_type,
            "Braki_Suma_Pct": round(missing_pct, 2),
            "Unikalnych_Wartosci": unique_count,
            "Najczestsza_Wartosc": str(top_value),
            "Dominacja_Najczestszej_Pct": round(top_value_pct, 2),
            "Min": v_min,
            "Mediana": v_median,
            "Max": v_max,
            "Korelacja_z_Cena": round(v_cor_with_target, 3) if pd.notna(v_cor_with_target) else np.nan,
            "przyklady_Danych": sample_vals
        })
        
    return pd.DataFrame(report_rows)

# Wywołanie funkcji 
metadata_report = generate_metadata_table(df_almost_clean, target_col="price")

display(metadata_report.head())

# Zapis do pliku Excel
metadata_report.to_excel("Raport_Zmiennych_Otomoto.xlsx", index=False)

In [ ]:
df_clean = df_almost_clean.copy()
# Zmiany typów danych, przeliczanie eur na pln, usunięcie jednostek
kurs_eur = 4.6129

# Przeliczanie walut
is_eur = df_clean['currency'] == "EUR"
df_clean['price'] = np.where(is_eur, df_clean['price'] * kurs_eur, pd.to_numeric(df_clean['price'], errors='coerce'))
df_clean['currency'] = np.where(is_eur, "PLN", df_clean['currency'])

df_clean['number_of_doors'] = pd.to_numeric(df_clean['number_of_doors'], errors='coerce')
df_clean['number_of_seats'] = pd.to_numeric(df_clean['number_of_seats'], errors='coerce')

# Uniwersalna funkcja do czyszczenia tekstu i konwersji na liczby float
def clean_numeric_col(series, extra_strip=""):
    # 1. Usuniecie wszystkich białe znaki i zamieniamy przecinki na kropki
    s = series.astype(str).str.replace(r"\s+", "", regex=True).str.replace(",", ".", regex=False)
    
    if extra_strip:
        s = s.str.replace(extra_strip, "", case=False, regex=False)
        
    # 3. Dodatkowo usuwamy jednostki spalania
    s = s.str.replace("l/100km", "", case=False, regex=False)
    s = s.str.replace("kWh/100km", "", case=False, regex=False)
    
    # 4. Bezpieczna konwersja na float
    return pd.to_numeric(s, errors='coerce')

# Wywołanie funkcji z poprawionymi jednostkami
df_clean['mileage'] = clean_numeric_col(df_clean['mileage'], "km")
df_clean['engine_displacement'] = clean_numeric_col(df_clean['engine_displacement'], "cm3")
df_clean['power'] = clean_numeric_col(df_clean['power'], "KM")

df_clean['fuel_consumption_in_city'] = clean_numeric_col(df_clean['fuel_consumption_in_city'])
df_clean['average_consumption'] = clean_numeric_col(df_clean['average_consumption'])
df_clean['combustion_in_the_mixed_cycle'] = clean_numeric_col(df_clean['combustion_in_the_mixed_cycle'])
df_clean['burning_outside_the_city'] = clean_numeric_col(df_clean['burning_outside_the_city'])
df_clean['co2_emissions'] = clean_numeric_col(df_clean['co2_emissions'], "g/km")



print("Typy wybranych kolumn:")
print(df_clean[['price', 'mileage', 'engine_displacement', 'power', 'number_of_doors']].dtypes)
print("\nPróbka zmienionych kolumn numerycznych:")
display(df_clean[['price', 'currency', 'mileage', 'engine_displacement', 'power', 'fuel_consumption_in_city']].head(3))

In [ ]:
# Zapisujemy liczbę wierszy przed operacją
wiersze_przed = len(df_clean)

# Usunięcie dokładnie powtarzających się wierszy
df_clean.drop_duplicates(inplace=True)

# Zapisujemy liczbę wierszy po operacji
wiersze_po = len(df_clean)

print(f"Liczba wierszy przed oczyszczeniem: {wiersze_przed}")
print(f"Liczba wierszy po oczyszczeniu:     {wiersze_po}")
print(f"Usunięto łącznie duplikatów:        {wiersze_przed - wiersze_po} wierszy.")

In [ ]:
vars_passive_safety = [
    "passenger_airbag", "driver_airbag", "side_airbags_front", "isofix_child_seat_attachment_points", 
    "air_curtains_front", "driver_side_airbag", "air_curtains_rear", "passenger_knee_airbag", 
    "driver's_knee_pad", "side_airbags_rear", "accident_notification_system", 
    "a_system_that_minimizes_the_effects_of_a_collision", "central_airbag", "front_airbags_rear", 
    "rear_curtain_airbag_crosswise", "rear_seat_belt_airbag", "rollover_protection_system", 
    "collision_protection_system_rear", "collision_protection_system_side", "collision_hearing_protection_system"
]
vars_active_safety_adas = [
    "abs", "esp", "power_steering", "traction_control", "brake_assist_system", 
    "electronic_tire_pressure_control", "hill_start_assist_hill_holder", "electronic_brake_force_distribution", 
    "speed___limiter", "lane_assist_lane_change_control", "brake_assist_brake_assist", 
    "traffic_sign_recognition_system", "collision_warning_system", "blind_spot_assistant_sensor", 
    "acc_adaptive_cruise_control", "distance_control_to_the_vehicle_in_front", "lane_assistant", 
    "active_emergency_braking_assist", "automatic_descent_control", "driver_fatigue_detection_system", 
    "emergency_braking_system_for_pedestrian_protection", "city_emergency_braking_assistant", 
    "active_lane_change_assist", "active_speed_limit_sign_recognition", "intersection_collision_prevention_assistant", 
    "traffic_jam_assistant", "cornering_assistant", "active_driver_fitness_monitoring", 
    "autonomous_steering_system", "cruise_control", "break_recommendation_system_during_the_route"
]
vars_comfort = [
    "electric_front_windows", "electrically_adjustable_side_mirrors", "heated_side_mirrors", 
    "electric_rear_windows", "armrests_front", "rain_sensor", "dusk_sensor", "heated_driver's_seat", 
    "heated_passenger_seat", "tinted_rear_windows", "armrests_rear", "automatic_air_conditioning", 
    "electrically_folding_side_mirrors", "automatic_two-zone_air_conditioning", 
    "electrically_adjustable_driver's_seat", "wipers", "electric_parking_brake", 
    "electrically_adjustable_passenger_seat", "regulation_electric_lumbar_support_driver", 
    "starting_the_engine_without_using_the_keys", "heated_windscreen", 
    "regulation_electric_lumbar_support_passenger", "memory_seat", "heated_steering_wheel", 
    "keyless_entry", "air_conditioning_for_rear_passengers", "heated_rear_seats", 
    "electrically_adjustable_steering_column", "electric_roof", "glass_sunroof_sliding_and_tilting_electric", 
    "panoramic_roof", "ventilated_front_seats", "automatic_heating_control", 
    "blinds_on_the_side_windows_lowered_manually", "automatic_air_conditioning_3_zones", "auxiliary_heating", 
    "front_seats_with_massage_functions", "automatic_air_conditioning_4_or_more_zones", 
    "the_rear_window_blind_is_electrically_adjustable", "the_roof_opens_electrically_with_a_remote_control", 
    "manual_air_conditioning", "keyless_go", "digital_key","start/stop_system", "energy_recovery_system", "charging_cable","particulate_filter"
]
vars_multimedia = [
    "radio", "usb_socket", "multifunction_steering_wheel", "bluetooth_interface", 
    "steering_wheel_with_radio_controls", "loudspeeker_sysetm", "satellite_navigation_system", 
    "sound_system", "touchscreen", "android_auto", "apple_carplay", "internet_access", 
    "control_vehicle_functions_with_your_voice", "wireless_charging_of_devices", "head-up_display"
]
vars_parking = [
    "rear_distance_control_when_parking", "rear_parking_camera", "distance_control_in_front_when_parking", 
    "park_assistant_parking_assistant", "camera_in_the_side_mirror", "360_panoramic_camera", 
    "rear_cross_traffic_alert", "independent_parking_system"
]
vars_lighting = [
    "fog_lamps", "daytime_running_lights", "led_daytime_running_lights", "rear_lamps_in_led_technology", 
    "front_lamps_in_led_technology", "cornering_lamps", "headlight_washers", "lighting_the_way_home", 
    "high_beam_assistant", "led_interior_lighting", "adaptive_lighting", "fog_lamps_in_led_technology", 
    "bi-xenon_lamps", "dynamic_cornering_lights", "xenon_lamps"
]
vars_design_interior = [
    "leather_steering_wheel", "fabric_upholstery", "leather-trimmed_gear_lever", "leather_upholstery", 
    "sports_steering_wheel", "sports_seats_front", "shifting_gears_on_the_steering_wheel", 
    "partial_leather_upholstery", "alcantara_upholstery"
]
vars_wheels_chassis = [
    "steel_wheels", "summer_tires", "sport_suspension", "comfort_suspension", "differential_lock", 
    "run-flat_tires", "multigrade_tires", "electronic_regulation_suspension_characteristics", 
    "adjustable_suspension", "hydropneumatic_suspension", "air_suspension", "winter_tires"
]

all_157_vars = (vars_passive_safety + vars_active_safety_adas + vars_comfort + vars_multimedia + 
                vars_parking + vars_lighting + vars_design_interior + vars_wheels_chassis)


print(f"Liczba zadeklarowanych zmiennych wyposażenia: {len(all_157_vars)}")

In [ ]:
df_clean['car_age'] = 2023 - pd.to_numeric(df_clean['production_year'], errors='coerce')
prawda_wartosci = ["True", "Tak", True, "TRUE"]

for col in all_157_vars:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].isin(prawda_wartosci).astype(int)

# Liczenie punktów
df_clean['score_passive_safety'] = df_clean[[c for c in vars_passive_safety if c in df_clean.columns]].sum(axis=1)
df_clean['score_active_safety'] = df_clean[[c for c in vars_active_safety_adas if c in df_clean.columns]].sum(axis=1)
df_clean['score_comfort'] = df_clean[[c for c in vars_comfort if c in df_clean.columns]].sum(axis=1)
df_clean['score_multimedia'] = df_clean[[c for c in vars_multimedia if c in df_clean.columns]].sum(axis=1)
df_clean['score_parking'] = df_clean[[c for c in vars_parking if c in df_clean.columns]].sum(axis=1)
df_clean['score_lighting'] = df_clean[[c for c in vars_lighting if c in df_clean.columns]].sum(axis=1)
df_clean['score_design_interior'] = df_clean[[c for c in vars_design_interior if c in df_clean.columns]].sum(axis=1)


kolumny_sprawdzajace = ['car_age', 'score_passive_safety', 'score_active_safety', 'score_comfort', 'score_multimedia', 'score_parking', 'score_lighting', 'score_design_interior']
print(f"Statystyki nowo utworzonych kolumn:")
display(df_clean[kolumny_sprawdzajace].describe())

In [ ]:
# liczba wystąpień oraz procenty dla każdej kategorii paliwa w danych 
liczba = df_clean['fuel_type'].value_counts()
procenty = df_clean['fuel_type'].value_counts(normalize=True) * 100

fuel_stats = pd.DataFrame({
    'Liczba ogłoszeń': liczba,
    'Procent (%)': procenty.round(2) # Zaokrąglamy do 2 miejsc po przecinku
})

display(fuel_stats)

In [ ]:
# 1. Sprawdzenie flagi stanu pojazdu ('state')
if 'state' in df_clean.columns:
    # Wyliczamy liczbę wystąpień oraz procenty
    state_counts = df_clean['state'].value_counts()
    state_percentages = df_clean['state'].value_counts(normalize=True) * 100
    
    state_stats = pd.DataFrame({
        'Liczba ogłoszeń': state_counts,
        'Procent (%)': state_percentages.round(2)
    })
    
    display(state_stats)
    
    nowe_count = state_counts.get('New', 0)
    uzywane_count = state_counts.get('Used', 0)
    
    print(f"\nPodsumowanie:")
    print(f"Liczba aut oznaczonych jako 'New': {nowe_count}")
    print(f"Liczba aut oznaczonych jako 'Used': {uzywane_count}")



In [ ]:
# usuniecie aut nowych 
df_clean = df_clean[df_clean['state'] == 'Used']
print(f"\nUsunięto auta nowe. Pozostało wierszy w bazie: {len(df_clean)}")

paliwa_do_usuniecia = ["Gasoline + CNG", "Hydrogen"]

if 'fuel_type' in df_clean.columns:
    wiersze_przed_paliwami = len(df_clean)
    
    df_clean = df_clean[~df_clean['fuel_type'].isin(paliwa_do_usuniecia)]
    
    wiersze_po_paliwach = len(df_clean)
    print(f"Usunięto {wiersze_przed_paliwami - wiersze_po_paliwach} wierszy z powodu rzadkiego rodzaju paliwa ({', '.join(paliwa_do_usuniecia)}).")

usun_ogolne = [
    "category", "color", "id", "location", "offer_creation_date", "offer_from", "offer_title", 
    "seller_registration_year", "number_of_doors", "number_of_seats", "show_offers_with_vin_number", 
    "generation", "color_type", "version", "fuel_consumption_in_city", 
    "date_of_first_registration_in_vehicle_history", "financing_possibility", "vehicle_registration_number", 
    "co2_emissions", "manufacturer's_warranty_period", "or_to_mileage_km", "combustion_in_the_mixed_cycle", 
    "monthly_rate", "burning_outside_the_city", "autonomy", "type_of_battery_property", 
    "number_of_remaining_installments", "battery_capacity", "initial_charge", "redemption_value", 
    "truck_homologation", "registered_as_a_monument", "average_consumption", "charging_time", 
    "has_a_registration_number", "state"
]
usun_bo_malo = [
    "ventilated_rear_seats", "fast_charging_function", "fabric_roof", "engine_sound_system_avas", 
    "tuning", "adjustable_center_differential", "rear_windshield_windstop", "pcc_predictive_cruise_control", 
    "the_second_glass_sunroof_sliding_and_tilting_el", "blinds_on_the_side_windows_lowered_electrically", 
    "external_wrapping", "hardtop", "rear_seats_with_massage_functions", "laser_reflectors", 
    "ceramic_composite_brakes", "plastic_roof", "right_hand_drive_english", "front_piping", 
    "off-road_tires", "glass_sunroof_sliding_and_tilting_manually"
]
usun_chyba = [
    "alloy_wheels_16", "alloy_wheels_18", "aluminum_rims_17", "alloy_wheels_19", 
    "aluminum_rims_20", "aluminum_rims_15", "alloy_wheels_from_21"
]
usun_fin = ["vat", "vat_margin", 'leasing']
usun_po_przeliczeniu = ["currency", "production_year"]

wszystkie_do_usuniecia = all_157_vars + usun_ogolne + usun_bo_malo + usun_chyba + usun_fin + usun_po_przeliczeniu

shape_przed = df_clean.shape
cols_to_drop = [col for col in wszystkie_do_usuniecia if col in df_clean.columns]
df_clean.drop(columns=cols_to_drop, inplace=True)

print(f"Kolumny przed usunięciem: {shape_przed[1]}")
print(f"Kolumny po usunięciu: {df_clean.shape[1]}")
print(f"Usunięto {shape_przed[1] - df_clean.shape[1]} kolumn.")


In [ ]:
# 1. Obliczanie liczby i procentu braków danych dla każdej kolumny
df_do_analizy = df_clean  
liczba_brakow = df_do_analizy.isna().sum()
procent_brakow = (liczba_brakow / len(df_do_analizy)) * 100

# 2. Tworzenie tabeli podsumowującej
raport_brakow = pd.DataFrame({
    'Liczba braków': liczba_brakow,
    'Procent (%)': procent_brakow
})

# 3. Filtracja kolumn i sortowanie według procentu braków danych
raport_brakow = raport_brakow[raport_brakow['Liczba braków'] > 0].sort_values(by='Procent (%)', ascending=False)

# 4. Generowanie wykresu
if len(raport_brakow) > 0:
    top_n = 30
    raport_wykres = raport_brakow.head(top_n)
    
    # Słownik tłumaczący angielskie zmienne na język polski
    slownik_brakow = {
        'dealership_warranty_included': 'Gwarancja dealerska',
        'damaged': 'Uszkodzony',
        'first_owner_from_new': 'Pierwszy właściciel',
        'serviced_at_aso': 'Serwisowany w ASO',
        'no_accident': 'Bezwypadkowy',
        'registered_in_poland': 'Zarejestrowany w Polsce',
        'country_of_origin': 'Kraj pochodzenia',
        'drive_type': 'Typ napędu',
        'transmission': 'Skrzynia biegów',
        'engine_displacement': 'Pojemność silnika',
        'power': 'Moc silnika',
        'mileage': 'Przebieg',
        'vehicle_brand': 'Marka pojazdu',
        'vehicle_model': 'Model pojazdu'
    }
    
    etykiety_y = [slownik_brakow.get(col, col) for col in raport_wykres.index]
    
    plt.figure(figsize=(12, max(6, len(raport_wykres) * 0.4)))
    
    
    ax = sns.barplot(
        x=raport_wykres['Procent (%)'], 
        y=etykiety_y, 
        hue=etykiety_y,       
        palette='Reds_r',     
        legend=False          
    )
    
    # Dodanie dokładnych wartości procentowych na końcu każdego paska
    for p in ax.patches:
        width = p.get_width()
        if width > 0:
            ax.text(width + 0.5, p.get_y() + p.get_height() / 2., 
                    f'{width:.1f}%', 
                    ha="left", va="center", fontsize=10, color='black')
        
    
    plt.xlabel("Odsetek braków danych (%)", fontsize=11)
    plt.ylabel("")
    
    plt.xlim(0, min(100, raport_wykres['Procent (%)'].max() + 10))
    
    plt.tight_layout()
    plt.show()
    
    print(f"Łącznie w zbiorze danych jest {len(raport_brakow)} kolumn zawierających braki (NaN).")
else:
    print("W wybranej ramce danych nie ma już żadnych braków danych (0 NaN)!")

In [ ]:
# ile jest braków przed imputacją
print("SPRAWDZENIE PRZED IMPUTACJĄ")
kluczowe_kolumny = ['drive_type', 'transmission', 'engine_displacement', 'power', 'mileage']
print(df_clean[kluczowe_kolumny].isna().sum())

def znajdz_mode(x):
    m = x.dropna().mode()
    return m.iloc[0] if not m.empty else np.nan

gb = df_clean.groupby(['vehicle_brand', 'vehicle_model'])

df_clean['drive_type'] = df_clean['drive_type'].fillna(gb['drive_type'].transform(znajdz_mode))
df_clean['transmission'] = df_clean['transmission'].fillna(gb['transmission'].transform(znajdz_mode))
df_clean['engine_displacement'] = df_clean['engine_displacement'].fillna(gb['engine_displacement'].transform('median'))
df_clean['power'] = df_clean['power'].fillna(gb['power'].transform('median'))

wiersze_przed = len(df_clean)


wiersze_do_usuniecia = df_clean[df_clean[kluczowe_kolumny].isna().any(axis=1)]

print("\nANALIZA AUT PRZEZNACZONYCH DO USUNIĘCIA")
if len(wiersze_do_usuniecia) > 0:
  
    print("Podział usuwanych aut według rodzaju paliwa:")
    print(wiersze_do_usuniecia['fuel_type'].value_counts())
    

    ile_elektrykow = (wiersze_do_usuniecia['fuel_type'] == 'Electric').sum()
    procent_elektrykow = (ile_elektrykow / len(wiersze_do_usuniecia)) * 100
    print(f"\n-> Usunięte elektryki: {ile_elektrykow} szt. ({procent_elektrykow:.2f}% wszystkich usuwanych wierszy)")
else:
    print("Brak wierszy do usunięcia.")

df_clean.dropna(subset=kluczowe_kolumny, inplace=True)


print("\nSPRAWDZENIE PO IMPUTACJI I USUŃ. NA")
print(df_clean[kluczowe_kolumny].isna().sum())
print(f"Liczba wierszy usuniętych z powodu nieodzyskalnych braków danych: {wiersze_przed - len(df_clean)}")

In [ ]:
print(f"Aktualny kształt: {df_clean.shape[0]} wierszy, {df_clean.shape[1]} kolumn.")

In [ ]:
df_encoded = df_clean.copy()

# kodowanie kraju pochodzenia z uwzględnieniem "reszty świata"
glowne_kraje = ["Poland", "Germany", "France", "United States"]

df_encoded['from_poland'] = (df_encoded['country_of_origin'] == "Poland").astype(int)
df_encoded['from_germany'] = (df_encoded['country_of_origin'] == "Germany").astype(int)
df_encoded['from_france'] = (df_encoded['country_of_origin'] == "France").astype(int)
df_encoded['from_usa'] = (df_encoded['country_of_origin'] == "United States").astype(int)

# Reszta z zagranicy
df_encoded['from_other_abroad'] = (~df_encoded['country_of_origin'].isin(glowne_kraje) & df_encoded['country_of_origin'].notna()).astype(int)

# Niewiadome pochodzenie
df_encoded['from_origin_unknown'] = df_encoded['country_of_origin'].isna().astype(int)


# Gwarancja dealerska
df_encoded['dealership_warranty_included'] = np.where(
    df_encoded['dealership_warranty_included'].isna() | df_encoded['dealership_warranty_included'].isin(["", "No"]), 
    0, 1
)
df_encoded.drop(columns=['country_of_origin'], inplace=True)

# Pozostałe zmienne binarne
zmienne_binarne = ["no_accident", "registered_in_poland", "serviced_at_aso", "first_owner_from_new", "damaged"]
for col in zmienne_binarne:
    if col in df_encoded.columns:
        df_encoded[col] = np.where(df_encoded[col].isna() | (df_encoded[col] == 0) | (df_encoded[col] == "0"), 0, 1)

# One-hot encoding dla zmiennych kategorycznych
cols_to_dummy = ["seller_type", "body_type", "fuel_type", "state", "transmission", "drive_type"]
cols_to_dummy = [c for c in cols_to_dummy if c in df_encoded.columns]
df_encoded = pd.get_dummies(df_encoded, columns=cols_to_dummy, dummy_na=False, dtype=int)


print(f"Nowy kształt danych (liczba kolumn znacznie wzrosła): {df_encoded.shape}")
# Wyciągamy kolumny pochodzenia, aby upewnić się, że suma w każdym wierszu wynosi dokładnie 1 
kolumny_pochodzenia = ['from_poland', 'from_germany', 'from_france', 'from_usa', 'from_other_abroad', 'from_origin_unknown']
print("Podgląd zakodowanych kolumn pochodzenia:")
display(df_encoded[kolumny_pochodzenia].head(5))

print(f"Czy każdy wiersz ma przypisany dokładnie jeden region?: {(df_encoded[kolumny_pochodzenia].sum(axis=1) != 1).sum()}")

In [ ]:
df_final = df_encoded.copy()

wlasne_granice = {
    'price': (1000, 1000000),
    'mileage': (0, 600000),
    'engine_displacement': (500, 6500), 
    'power': (20, 700),
    'car_age': (0, 50)
}

df_final_clean = df_final.copy()
wiersze_przed = len(df_final_clean)

print("POPRAWIONE CZYSZCZENIE (Z OCHRONĄ AUT ELEKTRYCZNYCH)\n")

for col, (v_min, v_max) in wlasne_granice.items():
    if col in df_final_clean.columns:
        if col == 'engine_displacement' and 'fuel_type_Electric' in df_final_clean.columns:
            # Maskujemy outliery, ale tylko dla aut, które nie są elektryczne 
            outliers_mask = ((df_final_clean[col] < v_min) | (df_final_clean[col] > v_max)) & (df_final_clean['fuel_type_Electric'] == 0)
        else:
            outliers_mask = (df_final_clean[col] < v_min) | (df_final_clean[col] > v_max)
            
        usunietych = outliers_mask.sum()
        df_final_clean = df_final_clean[~outliers_mask]
        
        print(f"• {col:<20}: Usunięto {usunietych:<5} wierszy poza zakresem [{v_min:,} - {v_max:,}]")

print("\nPODSUMOWANIE KOŃCOWE")
print(f"Liczba wierszy po bezpiecznym oczyszczeniu: {len(df_final_clean)}")

In [ ]:
print(f"Ostateczny kształt: {df_final.shape[0]} wierszy, {df_final.shape[1]} kolumn.")
print(f"Ostateczna liczba braków danych w całym DataFrame: {df_final.isna().sum().sum()}")
list(df_final.columns)

In [ ]:
print(f"Zapisywanie zbioru df_final_clean ({df_final_clean.shape[0]} wierszy, {df_final_clean.shape[1]} kolumn)")

# Zapis do pliku pickle
df_final_clean.to_pickle('samochody_clean.pkl')